# SQL SPARK

Codigo para reusar


### Conserva los resultados necesarios sin depender de las pestañas del editor

Reconoce las tablas, columnas y tipos en el contexto del Lakehouse

In [ ]:
SHOW TABLES IN dbo;

In [ ]:
DESCRIBE TABLE dbo.pedidos;

Selecciona, filtra y ordena pedidos con Spark SQL

In [ ]:
SELECT id_pedido, id_cliente, fecha_pedido, canal, importe_total
FROM dbo.pedidos
ORDER BY importe_total DESC, id_pedido
LIMIT 10;

In [ ]:
SELECT id_pedido, canal, importe_total
FROM dbo.pedidos
WHERE importe_total > 0
ORDER BY importe_total DESC, id_pedido
LIMIT 10;


Cuenta, agrega por canal y une pedidos con clientes

In [ ]:
SELECT 'clientes' AS tabla, COUNT(*) AS filas FROM dbo.clientes
UNION ALL
SELECT 'productos', COUNT(*) FROM dbo.productos
UNION ALL
SELECT 'pedidos', COUNT(*) FROM dbo.pedidos
UNION ALL
SELECT 'detalle_pedidos', COUNT(*) FROM dbo.detalle_pedidos;

In [ ]:
SELECT canal,
       COUNT(*) AS num_pedidos,
       SUM(importe_total) AS ventas_totales
FROM dbo.pedidos
GROUP BY canal
ORDER BY ventas_totales DESC;

In [ ]:
SELECT c.provincia,
       COUNT(*) AS num_pedidos,
       SUM(p.importe_total) AS ventas_totales
FROM dbo.pedidos AS p
INNER JOIN dbo.clientes AS c ON c.id_cliente = p.id_cliente
GROUP BY c.provincia
ORDER BY ventas_totales DESC;

Consulta los metadatos Delta e identifica numFiles

In [ ]:
DESCRIBE DETAIL dbo.pedidos;

Interpreta el historial y elige una versión disponible para consultar

In [ ]:
DESCRIBE HISTORY dbo.pedidos LIMIT 10;

In [ ]:
SELECT COUNT(*) AS filas_en_esa_version,
       SUM(importe_total) AS importe_en_esa_version
FROM dbo.pedidos VERSION AS OF <version_observada>;

In [ ]:
SELECT COUNT(*) AS filas_actuales,
       SUM(importe_total) AS importe_actual
FROM dbo.pedidos;

#

### Conserva los resultados necesarios sin depender de las pestañas del editor

Crea una consulta y cuenta las filas de cada tabla

In [ ]:
SELECT 'clientes' AS tabla, COUNT(*) AS filas FROM dbo.clientes
UNION ALL
SELECT 'productos', COUNT(*) FROM dbo.productos
UNION ALL
SELECT 'pedidos', COUNT(*) FROM dbo.pedidos
UNION ALL
SELECT 'detalle_pedidos', COUNT(*) FROM dbo.detalle_pedidos;

Consulta los diez pedidos más caros

In [ ]:
SELECT TOP 10 id_pedido, id_cliente, fecha_pedido, canal, importe_total
FROM dbo.pedidos
ORDER BY importe_total DESC;

Agrega ventas y pedidos por canal

In [ ]:
SELECT canal,
       COUNT(*) AS num_pedidos,
       SUM(importe_total) AS ventas_totales
FROM dbo.pedidos
GROUP BY canal
ORDER BY ventas_totales DESC;

Une pedidos y clientes para obtener ventas por provincia

In [ ]:
SELECT c.provincia,
       COUNT(*) AS num_pedidos,
       SUM(p.importe_total) AS ventas_totales
FROM dbo.pedidos AS p
INNER JOIN dbo.clientes AS c ON c.id_cliente = p.id_cliente
GROUP BY c.provincia
ORDER BY ventas_totales DESC;

Consulta los cinco productos con más unidades vendidas

In [ ]:
SELECT TOP 5 pr.nombre_producto,
       pr.categoria,
       SUM(d.cantidad) AS unidades_vendidas
FROM dbo.detalle_pedidos AS d
INNER JOIN dbo.productos AS pr ON pr.id_producto = d.id_producto
GROUP BY pr.nombre_producto, pr.categoria
ORDER BY unidades_vendidas DESC;

### Ejercicio: escribe con Spark SQL y contrasta el endpoint

Crea una tabla Delta exclusiva en lh_tienda desde Spark SQL

In [ ]:
CREATE TABLE dbo.practica_escritura_spark (
    id_practica INT,
    etiqueta VARCHAR(80)
)
USING DELTA;

Registra las filas y la última versión antes de insertar

In [ ]:
SELECT COUNT(*) AS filas_antes
FROM dbo.practica_escritura_spark;

In [ ]:
SELECT id_practica, etiqueta
FROM dbo.practica_escritura_spark
ORDER BY id_practica;

In [ ]:
DESCRIBE HISTORY dbo.practica_escritura_spark LIMIT 10;

In [ ]:
SELECT id_practica, etiqueta
FROM dbo.practica_escritura_spark
WHERE id_practica IN (1, 2)
   OR etiqueta IN ('spark-practica-01', 'endpoint-practica-01');

In [ ]:
INSERT INTO dbo.practica_escritura_spark (id_practica, etiqueta)
SELECT 1, 'spark-practica-01'
WHERE NOT EXISTS (
    SELECT 1
    FROM dbo.practica_escritura_spark
    WHERE id_practica = 1 OR etiqueta = 'spark-practica-01'
);

Comprueba la fila, el conteo y el historial después de insertar. Motor: Spark SQL.

In [ ]:
SELECT id_practica, etiqueta
FROM dbo.practica_escritura_spark
WHERE id_practica = 1 AND etiqueta = 'spark-practica-01';

In [ ]:
SELECT COUNT(*) AS filas_propias
FROM dbo.practica_escritura_spark
WHERE id_practica = 1 AND etiqueta = 'spark-practica-01';

In [ ]:
SELECT COUNT(*) AS filas_despues
FROM dbo.practica_escritura_spark;

In [ ]:
DESCRIBE HISTORY dbo.practica_escritura_spark LIMIT 10;

Lee la tabla desde el endpoint e intenta un INSERT para observar el rechazo

In [ ]:
SELECT id_practica, etiqueta
FROM dbo.practica_escritura_spark
WHERE id_practica = 1 AND etiqueta = 'spark-practica-01';

In [ ]:
INSERT INTO dbo.practica_escritura_spark (id_practica, etiqueta)
VALUES (2, 'endpoint-practica-01');

Vuelve a Spark y comprueba que el intento rechazado no escribió datos
Motor: Spark SQL, otra vez dentro del Lakehouse

In [ ]:
SELECT COUNT(*) AS filas_del_intento_rechazado
FROM dbo.practica_escritura_spark
WHERE id_practica = 2 OR etiqueta = 'endpoint-practica-01';

In [ ]:
SELECT id_practica, etiqueta
FROM dbo.practica_escritura_spark
WHERE id_practica = 1 AND etiqueta = 'spark-practica-01';

In [ ]:
SELECT COUNT(*) AS filas_tras_rechazo
FROM dbo.practica_escritura_spark;

In [ ]:
DESCRIBE HISTORY dbo.practica_escritura_spark LIMIT 10;

Borra solo la fila propia con Spark y comprueba los datos y el historial

In [ ]:
DELETE FROM dbo.practica_escritura_spark
WHERE id_practica = 1 AND etiqueta = 'spark-practica-01';

In [ ]:
SELECT COUNT(*) AS filas_propias_restantes
FROM dbo.practica_escritura_spark
WHERE id_practica = 1 AND etiqueta = 'spark-practica-01';

In [ ]:
SELECT COUNT(*) AS filas_finales
FROM dbo.practica_escritura_spark;

In [ ]:
DESCRIBE HISTORY dbo.practica_escritura_spark LIMIT 10;

### Ejercicio: historial, time travel y clon Delta con Spark SQL

Crea una tabla Delta propia y registra sus dos filas iniciales

In [ ]:
SHOW TABLES IN dbo;

In [ ]:
CREATE TABLE dbo.practica_historial_delta (
  id INT,
  importe_centimos INT
) USING DELTA;

In [ ]:
SET TIME ZONE 'UTC';

In [ ]:
INSERT INTO dbo.practica_historial_delta
VALUES (1, 1000), (2, 2000);

In [ ]:
SELECT * FROM dbo.practica_historial_delta ORDER BY id;

Actualiza y borra una fila, interpretando las versiones del historial

In [ ]:
DESCRIBE HISTORY dbo.practica_historial_delta LIMIT 20;

In [ ]:
SELECT date_format(current_timestamp(), 'yyyy-MM-dd HH:mm:ss.SSS') AS instante_utc;

In [ ]:
UPDATE dbo.practica_historial_delta
SET importe_centimos = 1500
WHERE id = 1;

In [ ]:
SELECT * FROM dbo.practica_historial_delta ORDER BY id;

In [ ]:
DELETE FROM dbo.practica_historial_delta WHERE id = 2;

In [ ]:
SELECT * FROM dbo.practica_historial_delta ORDER BY id;

In [ ]:
DESCRIBE HISTORY dbo.practica_historial_delta LIMIT 20;

Compara el estado actual con la versión inicial de dos filas

In [ ]:
SELECT id, importe_centimos
FROM dbo.practica_historial_delta VERSION AS OF <version_inicial>
ORDER BY id;

In [ ]:
SELECT id, importe_centimos
FROM dbo.practica_historial_delta
ORDER BY id;

Lee ese mismo estado por timestamp preciso y con zona horaria conocida

In [ ]:
SELECT id, importe_centimos
FROM dbo.practica_historial_delta
TIMESTAMP AS OF '<timestamp_inicial_UTC>'
ORDER BY id;

Crea un SHALLOW CLONE histórico y comprueba su historial propio

In [ ]:
CREATE TABLE dbo.practica_historial_clon
SHALLOW CLONE dbo.practica_historial_delta VERSION AS OF <version_inicial>;

In [ ]:
SELECT * FROM dbo.practica_historial_clon ORDER BY id;

In [ ]:
DESCRIBE HISTORY dbo.practica_historial_clon LIMIT 20;

 Cambia clon y origen por separado y contrasta que no se sincronizan

In [ ]:
UPDATE dbo.practica_historial_clon
SET importe_centimos = 9000
WHERE id = 1;

In [ ]:
SELECT * FROM dbo.practica_historial_clon ORDER BY id;

In [ ]:
SELECT * FROM dbo.practica_historial_delta ORDER BY id;

In [ ]:
INSERT INTO dbo.practica_historial_delta VALUES (3, 3000);

In [ ]:
SELECT * FROM dbo.practica_historial_delta ORDER BY id;

In [ ]:
SELECT * FROM dbo.practica_historial_clon ORDER BY id;

Guarda evidencias y conserva los objetos o retira primero el clon y después su origen

In [ ]:
DROP TABLE dbo.practica_historial_clon;

In [ ]:
DROP TABLE dbo.practica_historial_delta;